In [ ]:
import os
from haystack import Pipeline
from haystack_integrations.document_stores.pinecone import PineconeDocumentStore
from pathlib import Path
from haystack.components.converters import PyPDFToDocument
from haystack.components.preprocessors import DocumentSplitter
from haystack_integrations.components.embedders.google_genai import (
    GoogleGenAIDocumentEmbedder,
    GoogleGenAITextEmbedder,
)
from dotenv import load_dotenv
from haystack_integrations.components.retrievers.pinecone import (
    PineconeEmbeddingRetriever,
)
from haystack.components.generators import OpenAIGenerator
from haystack.utils import Secret
from haystack.components.builders import PromptBuilder
from haystack_integrations.components.connectors.langfuse import LangfuseConnector

d:\ProdRAG\prodRAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [59]:
load_dotenv()

True

In [ ]:
converter = PyPDFToDocument()

docs = converter.run(sources=[Path(r"../example.pdf")])["documents"]

In [25]:
print(docs)

[Document(id=d7dea9aaecbb32cd18daef5dd9453cec15693357857034cc98eb00203300d813, content: 'Course Proposal: Blockchain Technology and Its Applications in Modern 
Computing 
1. Introduction 
W...', meta: {'file_path': 'Blockchain_Course_Proposal.pdf'})]


In [ ]:
splitter = DocumentSplitter(
    split_by="word",
    split_length=10,
    split_overlap=0,
)

In [ ]:
docs_split = splitter.run(documents=docs)["documents"]

In [28]:
print(docs_split)

[Document(id=3ee6fc61b4b25e584b9babaf2c9cb5068c6ca16e16510e386279c5881e6680dc, content: 'Course Proposal: Blockchain Technology and Its Applications in Modern 
Computing ', meta: {'file_path': 'Blockchain_Course_Proposal.pdf', 'source_id': 'd7dea9aaecbb32cd18daef5dd9453cec15693357857034cc98eb00203300d813', 'page_number': 1, 'split_id': 0, 'split_idx_start': 0}), Document(id=b3063b25569899cfaa06108fc311dcdcd1e57f5f1c734e6a7b388f63b8177698, content: '
1. Introduction 
With the rapid advancement of decentralized technologies, Blockchain ', meta: {'file_path': 'Blockchain_Course_Proposal.pdf', 'source_id': 'd7dea9aaecbb32cd18daef5dd9453cec15693357857034cc98eb00203300d813', 'page_number': 1, 'split_id': 1, 'split_idx_start': 81}), Document(id=4dcd615e72bfc560f978c44dcb68950fdc7eb1f2e0a68473558818c653e6c5f9, content: 'has emerged as a 
transformative force across industries. From finance ', meta: {'file_path': 'Blockchain_Course_Proposal.pdf', 'source_id': 'd7dea9aaecbb32cd18daef5dd9453cec15

In [ ]:
document_store = PineconeDocumentStore(
    index="practice",
    metric="cosine",
    dimension=3072,
    spec={"serverless": {"region": "us-east-1", "cloud": "aws"}},
)

In [30]:
embedder = GoogleGenAIDocumentEmbedder(api="gemini")

In [ ]:
documents_with_embeddings = embedder.run(docs_split)["documents"]

Calculating embeddings: 3it [00:17,  5.96s/it]


In [32]:
document_store.write_documents(documents_with_embeddings)

Upserted vectors: 100%|██████████| 94/94 [02:27<00:00,  1.57s/it]


94

In [48]:
prompt_template = """
According to the contents:
{% for document in documents %}
{{document.content}}
{% endfor %}
Answer the given question: {{query}}
Answer:
"""

In [ ]:
prompt_builder = PromptBuilder(template=prompt_template)

PromptBuilder has 2 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.


In [ ]:
llm = OpenAIGenerator(
    api_key=Secret.from_env_var("GROQ_API_KEY"),
    api_base_url="https://api.groq.com/openai/v1",
    model="llama-3.1-8b-instant",
    generation_kwargs={"max_tokens": 512},
)

In [ ]:
query_pipeline = Pipeline()
query_pipeline.add_component("tracer", LangfuseConnector("Basic RAG Pipeline"))
query_pipeline.add_component("prompt", prompt_builder)
query_pipeline.add_component("llm", llm)
query_pipeline.add_component("text_embedder", GoogleGenAITextEmbedder())
query_pipeline.add_component(
    "retriever", PineconeEmbeddingRetriever(document_store=document_store)
)
query_pipeline.connect("text_embedder.embedding", "retriever.query_embedding")
query_pipeline.connect("retriever.documents", "prompt.documents")
query_pipeline.connect("prompt", "llm")
query = "What are the main topics covered in the document?"
result = query_pipeline.run(
    {"text_embedder": {"text": query}, "prompt": {"query": query}}
)

Traces will not be logged to Langfuse because Haystack tracing is disabled. To enable, set the HAYSTACK_CONTENT_TRACING_ENABLED environment variable to true before importing Haystack.


In [ ]:
print(result["llm"]["replies"][0])

The main topics covered in the document appear to be:

1. Introduction to Blockchain technology
2. Cryptography Fundamentals 
3. Consensus Algorithms (specifically Proof of Work and Proof of Stake)
4. Practical Components of Blockchain technology

Additionally, the document mentions applications of blockchain technology in various fields, such as:

* Supply Chain
* Healthcare
* Finance

It also mentions the following specific components of blockchain technology:

* Hashing
* Public and private key encryption
* Digital signatures
* Distributed ledgers
* Mining
* Consensus algorithms
